# CORA vs Transformer — Benchmark Mínimo (T4)

| | Transformer | CORA |
|---|---|---|
| hidden_dim | 64 | 64 |
| enc layers | 2 | 2 |
| dec layers | 2 | 2 (doble cross-attention) |
| CRE | — | 2 capas × 2 iters |
| params | <2 M | <2 M |

2 000 steps · batch=1 · pre-tokenizado · sin MoE / VAL / Budget / ConvergenceGate

In [ ]:
import os, sys, time, math, random
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

# ── Device ────────────────────────────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    DEVICE  = xm.xla_device()
    USE_AMP = False
    print('[device] TPU/XLA')
except ImportError:
    if not torch.cuda.is_available():
        raise RuntimeError('\n⚠ No GPU.\nRuntime -> Change runtime type -> T4 GPU')
    DEVICE  = torch.device('cuda')
    USE_AMP = True
    torch.backends.cudnn.benchmark = True
    _free, _total = torch.cuda.mem_get_info()
    print(f'[device] {torch.cuda.get_device_name(0)}')
    print(f'[vram]   {_free/1e9:.1f} / {_total/1e9:.1f} GB libre')

torch.manual_seed(42)
random.seed(42)

In [ ]:
# ── Dataset sintético de razonamiento causal ──────────────────────────────────
CAUSE_EFFECT = [
    ('smoking',   'cancer'),  ('smoking',   'cough'),
    ('exercise',  'fitness'), ('exercise',  'fatigue'),
    ('stress',    'headache'),('stress',    'insomnia'),
    ('rain',      'flood'),   ('rain',      'growth'),
    ('drought',   'fire'),    ('cold',      'fever'),
    ('infection', 'fever'),   ('fever',     'thirst'),
    ('hunger',    'fatigue'), ('pollution', 'cough'),
    ('medicine',  'recovery'),('sleep',     'health'),
    ('heat',      'thirst'),  ('lightning', 'fire'),
    ('pressure',  'pain'),    ('sunlight',  'growth'),
    ('diet',      'health'),  ('noise',     'stress'),
    ('virus',     'infection'),('bacteria', 'infection'),
    ('fatigue',   'error'),   ('error',     'failure'),
]

def gen_dataset(n=2600, seed=42):
    random.seed(seed)
    pairs = (CAUSE_EFFECT * (n // len(CAUSE_EFFECT) + 1))[:n]
    random.shuffle(pairs)
    data = []
    for cause, effect in pairs:
        r = random.random()
        if r < 0.25:
            data.append((f'does {cause} cause {effect}',
                         f'yes {cause} causes {effect}'))
        elif r < 0.50:
            data.append((f'what does {cause} cause',
                         f'{cause} causes {effect}'))
        elif r < 0.75:
            data.append((f'what causes {effect}',
                         f'{cause} causes {effect}'))
        else:
            wrong = random.choice([e for c, e in CAUSE_EFFECT if c != cause])
            data.append((f'does {cause} cause {wrong}',
                         f'no {cause} does not cause {wrong}'))
    random.shuffle(data)
    return data

ALL_DATA   = gen_dataset(2600)
TRAIN_DATA = ALL_DATA[:2000]
EVAL_DATA  = ALL_DATA[2000:2250]

# ── Vocabulario ───────────────────────────────────────────────────────────────
PAD_ID, BOS_ID, EOS_ID = 0, 1, 2
_counts = Counter()
for q, a in ALL_DATA:
    for tok in q.split() + a.split():
        _counts[tok] += 1
I2W        = ['<PAD>', '<BOS>', '<EOS>'] + [w for w, _ in _counts.most_common()]
W2I        = {w: i for i, w in enumerate(I2W)}
VOCAB_SIZE = len(W2I)
SKIP_IDS   = {PAD_ID, BOS_ID, EOS_ID}
print(f'[vocab] {VOCAB_SIZE} tokens')

# ── Longitudes ────────────────────────────────────────────────────────────────
MAX_SRC = max(len(q.split()) for q, _ in ALL_DATA) + 2
MAX_TGT = max(len(a.split()) for _, a in ALL_DATA) + 2
print(f'[seq]   MAX_SRC={MAX_SRC}  MAX_TGT={MAX_TGT}')

def encode(text, max_len):
    ids = [BOS_ID] + [W2I.get(w, PAD_ID) for w in text.split()] + [EOS_ID]
    ids = ids[:max_len]
    return ids + [PAD_ID] * (max_len - len(ids))

# ── Pre-tokenización completa (una sola vez, fuera del loop) ──────────────────
TRAIN_SRC = torch.tensor([encode(q, MAX_SRC) for q, _ in TRAIN_DATA], dtype=torch.long)
TRAIN_TGT = torch.tensor([encode(a, MAX_TGT) for _, a in TRAIN_DATA], dtype=torch.long)
EVAL_SRC  = torch.tensor([encode(q, MAX_SRC) for q, _ in EVAL_DATA],  dtype=torch.long)
EVAL_TGT  = torch.tensor([encode(a, MAX_TGT) for _, a in EVAL_DATA],  dtype=torch.long)
print(f'[data]  train={len(TRAIN_SRC)}  eval={len(EVAL_SRC)}')
print(f'[shape] SRC={tuple(TRAIN_SRC.shape)}  TGT={tuple(TRAIN_TGT.shape)}')

# ── Métrica word-F1 ───────────────────────────────────────────────────────────
def word_f1(pred, gold):
    p = [t for t in pred if t not in SKIP_IDS]
    g = [t for t in gold if t not in SKIP_IDS]
    if not p and not g: return 1.0
    if not p or  not g: return 0.0
    pc = Counter(p); gc = Counter(g)
    ov = sum((pc & gc).values())
    pr = ov / len(p); re = ov / len(g)
    return 2 * pr * re / (pr + re) if pr + re else 0.0

# ── Constantes globales de entrenamiento ──────────────────────────────────────
TOTAL_STEPS = 2000
EVAL_EVERY  = 500
print(f'[train] {TOTAL_STEPS} steps, eval every {EVAL_EVERY}')

In [ ]:
# ── Positional Encoding ───────────────────────────────────────────────────────
class PosEnc(nn.Module):
    def __init__(self, dim, max_len=64):
        super().__init__()
        pe  = torch.zeros(max_len, dim)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, x):  # x: [B, L, D]
        return x + self.pe[:x.shape[1]]

# ── Tiny Transformer (seq2seq estándar) ───────────────────────────────────────
class TinyTransformer(nn.Module):
    def __init__(self, vocab, dim=64, nhead=4, n_enc=2, n_dec=2, ffn=256):
        super().__init__()
        self.embed   = nn.Embedding(vocab, dim, padding_idx=PAD_ID)
        self.posenc  = PosEnc(dim)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dim, nhead, ffn, dropout=0.1, batch_first=True), n_enc)
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(dim, nhead, ffn, dropout=0.1, batch_first=True), n_dec)
        self.out = nn.Linear(dim, vocab, bias=False)
        self.out.weight = self.embed.weight   # weight tying

    def forward(self, src, tgt):
        # src/tgt: [B, L]
        T           = tgt.shape[1]
        causal_mask = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
        h_enc = self.posenc(self.embed(src))
        h_enc = self.encoder(h_enc, src_key_padding_mask=(src == PAD_ID))
        h_dec = self.posenc(self.embed(tgt))
        h_dec = self.decoder(h_dec, h_enc,
                             tgt_mask=causal_mask,
                             tgt_key_padding_mask=(tgt == PAD_ID),
                             memory_key_padding_mask=(src == PAD_ID))
        return self.out(h_dec)   # [B, T, vocab]

# ── CRE — Causal Reasoning Engine (inline minimal) ────────────────────────────
class CRELayer(nn.Module):
    """Una capa de message passing. Se comparte entre todas las iteraciones."""
    def __init__(self, dim=64, msg_dim=64):
        super().__init__()
        self.msg_fn  = nn.Sequential(
            nn.Linear(dim * 2, msg_dim), nn.GELU(), nn.Linear(msg_dim, msg_dim))
        self.gru     = nn.GRUCell(msg_dim, dim)
        self.norm    = nn.LayerNorm(dim)
        self.msg_dim = msg_dim

    def forward(self, h, src_idx, tgt_idx):
        # h: [N, D]   src_idx/tgt_idx: [E]
        n = h.shape[0]
        E = src_idx.shape[0]
        if E == 0:
            agg = torch.zeros(n, self.msg_dim, device=h.device, dtype=h.dtype)
        else:
            msgs = self.msg_fn(torch.cat([h[src_idx], h[tgt_idx]], dim=-1))   # [E, msg_dim]
            agg  = torch.zeros(n, self.msg_dim, device=h.device, dtype=msgs.dtype)
            agg.scatter_add_(0, tgt_idx.unsqueeze(-1).expand(-1, self.msg_dim), msgs)
        return self.norm(self.gru(agg, h.to(agg.dtype)))


class MinimalCRE(nn.Module):
    """CRE con weight sharing entre iteraciones."""
    def __init__(self, dim=64, msg_dim=64, n_layers=2, n_iters=2):
        super().__init__()
        self.layers  = nn.ModuleList([CRELayer(dim, msg_dim) for _ in range(n_layers)])
        self.n_iters = n_iters

    def forward(self, h, src_idx, tgt_idx):
        for _ in range(self.n_iters):
            for layer in self.layers:
                h = layer(h, src_idx, tgt_idx)
        return h


# ── GraphBuilder: encoder → nodos del grafo ───────────────────────────────────
class GraphBuilder(nn.Module):
    """Attention pooling: encoder output [L,D] -> N nodos [N,D]."""
    def __init__(self, dim=64, n_nodes=8):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_nodes, dim) * 0.02)
        self.k_proj  = nn.Linear(dim, dim, bias=False)
        self.v_proj  = nn.Linear(dim, dim, bias=False)
        # Grafo completamente conectado (buffer constante)
        src, tgt = zip(*[(i, j) for i in range(n_nodes)
                                 for j in range(n_nodes) if i != j])
        self.register_buffer('edge_src', torch.tensor(src, dtype=torch.long))
        self.register_buffer('edge_tgt', torch.tensor(tgt, dtype=torch.long))

    def forward(self, enc_out):   # enc_out: [L, D]  (sin batch)
        K = self.k_proj(enc_out)
        V = self.v_proj(enc_out)
        w = torch.softmax(
            self.queries @ K.T / math.sqrt(self.queries.shape[-1]), dim=-1)  # [N, L]
        return w @ V                                                          # [N, D]


# ── CORA Decoder Layer — doble cross-attention ────────────────────────────────
class CoraDecoderLayer(nn.Module):
    """Self-attn → cross-attn(encoder) → cross-attn(CRE) → FFN."""
    def __init__(self, dim=64, nhead=4, ffn=256):
        super().__init__()
        kw = dict(batch_first=True, dropout=0.1)
        self.sa  = nn.MultiheadAttention(dim, nhead, **kw)
        self.ca1 = nn.MultiheadAttention(dim, nhead, **kw)   # → encoder
        self.ca2 = nn.MultiheadAttention(dim, nhead, **kw)   # → CRE graph
        self.ffn = nn.Sequential(nn.Linear(dim, ffn), nn.GELU(), nn.Linear(ffn, dim))
        self.n1  = nn.LayerNorm(dim); self.n2 = nn.LayerNorm(dim)
        self.n3  = nn.LayerNorm(dim); self.n4 = nn.LayerNorm(dim)

    def forward(self, tgt, enc_mem, cre_mem, causal_mask, tgt_pad, src_pad):
        x, _ = self.sa( tgt, tgt, tgt, attn_mask=causal_mask, key_padding_mask=tgt_pad)
        tgt  = self.n1(tgt + x)
        x, _ = self.ca1(tgt, enc_mem, enc_mem, key_padding_mask=src_pad)
        tgt  = self.n2(tgt + x)
        x, _ = self.ca2(tgt, cre_mem, cre_mem)
        tgt  = self.n3(tgt + x)
        return self.n4(tgt + self.ffn(tgt))


# ── Tiny CORA ─────────────────────────────────────────────────────────────────
class TinyCORA(nn.Module):
    def __init__(self, vocab, dim=64, nhead=4, n_enc=2, n_dec=2, ffn=256,
                 n_nodes=8, cre_layers=2, cre_iters=2):
        super().__init__()
        self.embed   = nn.Embedding(vocab, dim, padding_idx=PAD_ID)
        self.posenc  = PosEnc(dim)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dim, nhead, ffn, dropout=0.1, batch_first=True), n_enc)
        self.graph   = GraphBuilder(dim, n_nodes)
        self.cre     = MinimalCRE(dim, dim, cre_layers, cre_iters)
        self.dec_layers = nn.ModuleList(
            [CoraDecoderLayer(dim, nhead, ffn) for _ in range(n_dec)])
        self.out = nn.Linear(dim, vocab, bias=False)
        self.out.weight = self.embed.weight   # weight tying

    def forward(self, src, tgt):
        # src/tgt: [1, L]
        T           = tgt.shape[1]
        causal_mask = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
        src_pad     = (src == PAD_ID)
        tgt_pad     = (tgt == PAD_ID)

        h_enc   = self.posenc(self.embed(src))
        h_enc   = self.encoder(h_enc, src_key_padding_mask=src_pad)   # [1, L, D]

        nodes   = self.graph(h_enc[0])                                 # [N, D]
        nodes   = self.cre(nodes, self.graph.edge_src, self.graph.edge_tgt)
        cre_mem = nodes.unsqueeze(0)                                   # [1, N, D]

        h_dec   = self.posenc(self.embed(tgt))
        for layer in self.dec_layers:
            h_dec = layer(h_dec, h_enc, cre_mem, causal_mask, tgt_pad, src_pad)

        return self.out(h_dec)   # [1, T, vocab]

# ── Verificación de parámetros ────────────────────────────────────────────────
with torch.no_grad():
    _tf = TinyTransformer(VOCAB_SIZE)
    _co = TinyCORA(VOCAB_SIZE)
n_tf = sum(p.numel() for p in _tf.parameters())
n_co = sum(p.numel() for p in _co.parameters())
print(f'[params] Transformer: {n_tf:,}   CORA: {n_co:,}')
assert n_tf < 2_000_000, f'Transformer demasiado grande: {n_tf:,}'
assert n_co < 2_000_000, f'CORA demasiado grande: {n_co:,}'
del _tf, _co


In [ ]:
# ── Transformer: entrenamiento ────────────────────────────────────────────────
tf_model  = TinyTransformer(VOCAB_SIZE).to(DEVICE)
tf_opt    = torch.optim.AdamW(tf_model.parameters(), lr=3e-4, weight_decay=1e-2)
tf_scaler = GradScaler(enabled=USE_AMP)
TF_LOSSES, TF_EVALS = [], []

def eval_tf():
    tf_model.eval()
    ls, ws = [], []
    with torch.no_grad():
        for i in range(min(100, len(EVAL_SRC))):
            src = EVAL_SRC[i:i+1].to(DEVICE)
            tgt = EVAL_TGT[i:i+1].to(DEVICE)
            with autocast(enabled=USE_AMP):
                logits = tf_model(src, tgt[:, :-1])
            ls.append(F.cross_entropy(logits.reshape(-1, VOCAB_SIZE),
                                      tgt[:, 1:].reshape(-1),
                                      ignore_index=PAD_ID).item())
            ws.append(word_f1(logits.argmax(-1).squeeze(0).tolist(),
                               tgt[0, 1:].tolist()))
    tf_model.train()
    return sum(ls)/len(ls), sum(ws)/len(ws)

_order = list(range(len(TRAIN_SRC)))
random.shuffle(_order)
t0 = time.perf_counter()
print('[Transformer] Training...')

for step in range(1, TOTAL_STEPS + 1):
    if (step - 1) % len(_order) == 0:
        random.shuffle(_order)
    i = _order[(step - 1) % len(_order)]

    src = TRAIN_SRC[i:i+1].to(DEVICE)
    tgt = TRAIN_TGT[i:i+1].to(DEVICE)

    tf_opt.zero_grad()
    with autocast(enabled=USE_AMP):
        logits = tf_model(src, tgt[:, :-1])
        loss   = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE),
                                 tgt[:, 1:].reshape(-1), ignore_index=PAD_ID)
    tf_scaler.scale(loss).backward()
    tf_scaler.unscale_(tf_opt)
    torch.nn.utils.clip_grad_norm_(tf_model.parameters(), 1.0)
    tf_scaler.step(tf_opt); tf_scaler.update()
    TF_LOSSES.append(loss.item())

    if step % EVAL_EVERY == 0 or step == TOTAL_STEPS:
        vl, vw = eval_tf()
        TF_EVALS.append((step, vl, vw))
        elapsed = time.perf_counter() - t0
        avg_l   = sum(TF_LOSSES[-EVAL_EVERY:]) / min(EVAL_EVERY, len(TF_LOSSES))
        print(f'  step={step:4d}  t={elapsed:.1f}s  train={avg_l:.4f}  val={vl:.4f}  wf1={vw:.3f}')

TF_TIME        = time.perf_counter() - t0
TF_FINAL_LOSS  = TF_EVALS[-1][1]
TF_FINAL_WF1   = TF_EVALS[-1][2]
TF_STEPS_PER_S = TOTAL_STEPS / TF_TIME
print(f'\n[Transformer] {TF_TIME:.1f}s  ({TF_STEPS_PER_S:.0f} steps/s)  wf1={TF_FINAL_WF1:.3f}')

In [ ]:
# ── CORA: entrenamiento ───────────────────────────────────────────────────────
co_model  = TinyCORA(VOCAB_SIZE).to(DEVICE)
co_opt    = torch.optim.AdamW(co_model.parameters(), lr=3e-4, weight_decay=1e-2)
co_scaler = GradScaler(enabled=USE_AMP)
CO_LOSSES, CO_EVALS = [], []

def eval_cora():
    co_model.eval()
    ls, ws = [], []
    with torch.no_grad():
        for i in range(min(100, len(EVAL_SRC))):
            src = EVAL_SRC[i:i+1].to(DEVICE)
            tgt = EVAL_TGT[i:i+1].to(DEVICE)
            with autocast(enabled=USE_AMP):
                logits = co_model(src, tgt[:, :-1])
            ls.append(F.cross_entropy(logits.reshape(-1, VOCAB_SIZE),
                                      tgt[:, 1:].reshape(-1),
                                      ignore_index=PAD_ID).item())
            ws.append(word_f1(logits.argmax(-1).squeeze(0).tolist(),
                               tgt[0, 1:].tolist()))
    co_model.train()
    return sum(ls)/len(ls), sum(ws)/len(ws)

_order = list(range(len(TRAIN_SRC)))
random.shuffle(_order)
t0 = time.perf_counter()
print('[CORA] Training...')

for step in range(1, TOTAL_STEPS + 1):
    if (step - 1) % len(_order) == 0:
        random.shuffle(_order)
    i = _order[(step - 1) % len(_order)]

    src = TRAIN_SRC[i:i+1].to(DEVICE)
    tgt = TRAIN_TGT[i:i+1].to(DEVICE)

    co_opt.zero_grad()
    with autocast(enabled=USE_AMP):
        logits = co_model(src, tgt[:, :-1])
        loss   = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE),
                                 tgt[:, 1:].reshape(-1), ignore_index=PAD_ID)
    co_scaler.scale(loss).backward()
    co_scaler.unscale_(co_opt)
    torch.nn.utils.clip_grad_norm_(co_model.parameters(), 1.0)
    co_scaler.step(co_opt); co_scaler.update()
    CO_LOSSES.append(loss.item())

    if step % EVAL_EVERY == 0 or step == TOTAL_STEPS:
        vl, vw = eval_cora()
        CO_EVALS.append((step, vl, vw))
        elapsed = time.perf_counter() - t0
        avg_l   = sum(CO_LOSSES[-EVAL_EVERY:]) / min(EVAL_EVERY, len(CO_LOSSES))
        print(f'  step={step:4d}  t={elapsed:.1f}s  train={avg_l:.4f}  val={vl:.4f}  wf1={vw:.3f}')

CO_TIME        = time.perf_counter() - t0
CO_FINAL_LOSS  = CO_EVALS[-1][1]
CO_FINAL_WF1   = CO_EVALS[-1][2]
CO_STEPS_PER_S = TOTAL_STEPS / CO_TIME
print(f'\n[CORA] {CO_TIME:.1f}s  ({CO_STEPS_PER_S:.0f} steps/s)  wf1={CO_FINAL_WF1:.3f}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# ── Tabla resumen ─────────────────────────────────────────────────────────────
_n_tf = sum(p.numel() for p in tf_model.parameters())
_n_co = sum(p.numel() for p in co_model.parameters())

print('=' * 55)
print(f'{"":22}{"Transformer":>16}{"CORA":>16}')
print('=' * 55)
print(f'{"Params":22}{_n_tf:>16,}{_n_co:>16,}')
print(f'{"Tiempo (s)":22}{TF_TIME:>16.1f}{CO_TIME:>16.1f}')
print(f'{"Steps/seg":22}{TF_STEPS_PER_S:>16.0f}{CO_STEPS_PER_S:>16.0f}')
print(f'{"Val loss":22}{TF_FINAL_LOSS:>16.4f}{CO_FINAL_LOSS:>16.4f}')
print(f'{"Val wF1":22}{TF_FINAL_WF1:>16.3f}{CO_FINAL_WF1:>16.3f}')
print('=' * 55)
_winner = 'CORA' if CO_FINAL_WF1 >= TF_FINAL_WF1 else 'Transformer'
print(f'Ganador (wF1): {_winner}')

# ── Curvas ────────────────────────────────────────────────────────────────────
def _smooth(x, w=100):
    return np.convolve(x, np.ones(w) / w, mode='valid') if len(x) >= w else np.array(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(_smooth(TF_LOSSES), label='Transformer', color='steelblue', alpha=0.85)
ax.plot(_smooth(CO_LOSSES), label='CORA',        color='coral',     alpha=0.85)
ax.set_title('Training Loss (smoothed w=100)')
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot([e[0] for e in TF_EVALS], [e[2] for e in TF_EVALS],
        'o-', label='Transformer', color='steelblue')
ax.plot([e[0] for e in CO_EVALS], [e[2] for e in CO_EVALS],
        'o-', label='CORA',        color='coral')
ax.set_title('Validation Word-F1')
ax.set_xlabel('Step'); ax.set_ylabel('wF1')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('benchmark_result.png', dpi=120, bbox_inches='tight')
plt.show()
print('[saved] benchmark_result.png')